# Demo 5: CLIP — Connecting Images and Language

**Before you start: go to Runtime → Change runtime type and select GPU.**

Every demo so far has used models trained on fixed categories — ImageNet's 
1,000 classes, COCO's 80 objects, VOC's 21 segments. To classify something new, 
you would need to collect labelled data and retrain.

**CLIP** (Contrastive Language-Image Pretraining, OpenAI 2021) breaks this constraint. 
It was trained on 400 million image-text pairs from the internet, learning to match 
images with their descriptions. The result is a model that understands both images 
and language in a shared space — so you can describe a category in plain English 
and the model understands what you mean, even if it has never seen a labelled 
example of that category.

This is called **zero-shot** learning: the model performs a new task without 
any task-specific training examples.

In this demo you will:
1. Understand how CLIP represents images and text in the same space
2. Use CLIP for zero-shot image classification
3. Search an image collection using natural language queries
4. Find the best text description for an image
5. Explore the limits of zero-shot understanding

## Setup

In [ ]:
!pip install -q transformers

In [ ]:
import torch
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import urllib.request
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load CLIP (ViT-B/32 variant — a good balance of speed and capability)
print('Loading CLIP model...')
model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
model.eval()
print('CLIP model loaded.')

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

## 1. How CLIP works

CLIP has two encoders — one for images and one for text — that map their 
inputs into a shared high-dimensional space. After training on 400 million 
image-text pairs, matching images and descriptions end up close together 
in this space, while mismatched pairs end up far apart.

The similarity between an image and a text description is measured by the 
**cosine similarity** of their embeddings — a number between -1 and 1, 
where 1 means perfectly aligned.

Let's see this in action:

In [ ]:
def get_image_embedding(image):
    """Encode an image into CLIP's shared embedding space."""
    inputs = processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        embedding = model.get_image_features(**inputs).pooler_output
    return F.normalize(embedding, dim=-1)   # L2 normalise


def get_text_embedding(text):
    """Encode a text description into CLIP's shared embedding space."""
    inputs = processor(text=text, return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        embedding = model.get_text_features(**inputs).pooler_output
    return F.normalize(embedding, dim=-1)   # L2 normalise


def cosine_similarity(a, b):
    """Compute cosine similarity between two normalised embedding vectors."""
    return (a * b).sum(dim=-1).item()


# Load a test image
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/klaverhenrik/Deep-Learning-for-Visual-Recognition-2026/main/data/Cute_dog.jpg',
    'dog.jpg'
)
dog_image = Image.open('dog.jpg').convert('RGB')

# Get embeddings
img_embedding = get_image_embedding(dog_image)
print(f'Image embedding shape: {img_embedding.shape}')   # (1, 512)

# Compare to different text descriptions
descriptions = [
    'a dog',
    'a cavalier king charles spaniel',
    'a fluffy dog in a field',
    'a cat',
    'a car',
    'a beautiful landscape',
]

print('\nCosine similarity between the dog image and various descriptions:')
print('-' * 55)
for desc in descriptions:
    txt_emb = get_text_embedding(desc)
    sim     = cosine_similarity(img_embedding, txt_emb)
    bar     = '█' * int(sim * 40)
    print(f'{desc:35s}  {sim:.3f}  {bar}')

The matching descriptions get higher similarity scores. Notice that 
'a cavalier king charles spaniel scores higher than just 'a dog' — CLIP understands 
breed-level specificity. And 'a car' or 'a cat' score much lower, 
even though both are common ImageNet categories.

## 2. Zero-shot image classification

Zero-shot classification works by comparing the image embedding to 
the text embedding of each candidate class label. The class whose 
text description is most similar to the image wins.

Crucially, the classes do not need to be predefined — you can describe 
any category in plain English.

In [ ]:
def zero_shot_classify(image, class_names, prompt_template='a photo of a {}'):
    """
    Classify an image into one of the given classes using CLIP.
    No training required — just describe the classes in plain English.
    
    Args:
        image:            PIL image
        class_names:      list of class name strings
        prompt_template:  text template; {} is replaced with the class name
    """
    # Encode image
    img_emb = get_image_embedding(image)   # (1, 512)
    
    # Encode all class descriptions
    prompts  = [prompt_template.format(name) for name in class_names]
    txt_embs = get_text_embedding(prompts)  # (n_classes, 512)
    
    # Compute similarity scores and convert to probabilities
    similarities = (img_emb @ txt_embs.T).squeeze(0)   # (n_classes,)
    probs        = F.softmax(similarities * 100, dim=0).cpu().numpy()
    
    return list(zip(class_names, probs))


def show_classification(image, predictions, title=None):
    """Display image with classification bar chart."""
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4))
    ax_img.imshow(image)
    ax_img.axis('off')
    if title:
        ax_img.set_title(title)

    labels = [p[0] for p in predictions]
    probs  = [p[1] * 100 for p in predictions]
    colors = ['#2196F3' if i == 0 else '#90CAF9'
              for i in range(len(labels))]

    ax_bar.barh(labels[::-1], probs[::-1], color=colors[::-1])
    ax_bar.set_xlabel('Confidence (%)')
    ax_bar.set_title('Zero-shot classification')
    ax_bar.set_xlim(0, 100)
    for i, (label, prob) in enumerate(zip(labels[::-1], probs[::-1])):
        ax_bar.text(prob + 0.5, i, f'{prob:.1f}%', va='center')

    plt.tight_layout()
    plt.show()


# Standard animal classification
classes = ['dog', 'cat', 'bird', 'horse', 'fish', 'rabbit']
preds   = zero_shot_classify(dog_image, classes)
preds.sort(key=lambda x: x[1], reverse=True)
show_classification(dog_image, preds, title='Standard animal classes')

In [ ]:
# Now try classes that would never appear in a standard dataset
# CLIP handles these because it understands language, not just fixed labels
custom_classes = [
    'a happy dog',
    'an angry dog',
    'a dog outdoors',
    'a dog indoors',
    'a dog running',
    'a dog sitting still',
]
preds = zero_shot_classify(dog_image, custom_classes, prompt_template='{}')
preds.sort(key=lambda x: x[1], reverse=True)
show_classification(dog_image, preds, title='Custom descriptive classes — no training needed')

In [ ]:
# Try a domain-specific classification — medical-style descriptions
# (Note: this is a demo only — do not use for real medical decisions)
urllib.request.urlretrieve(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Dog_Breeds.jpg/320px-Dog_Breeds.jpg',
    'dogs.jpg'
)
dogs_image = Image.open('dogs.jpg').convert('RGB')

breed_classes = [
    'labrador retriever', 'german shepherd', 'golden retriever',
    'french bulldog', 'poodle', 'beagle', 'rottweiler', 'husky'
]
preds = zero_shot_classify(dogs_image, breed_classes)
preds.sort(key=lambda x: x[1], reverse=True)
show_classification(dogs_image, preds, title='Dog breed classification — zero shot')

## 3. Text-to-image retrieval

Given a collection of images and a text query, CLIP can find the images 
that best match the query — without any labels on the images.

This is essentially a **search engine** for images, driven by natural language. 
You describe what you are looking for, and CLIP finds the closest matches 
in the collection.

In [ ]:
# Build a small image collection to search over
image_collection = [
    ('golden retriever',   'https://upload.wikimedia.org/wikipedia/commons/thumb/b/bd/Golden_Retriever_Dukedestiny01_drvd.jpg/320px-Golden_Retriever_Dukedestiny01_drvd.jpg'),
    ('cat on windowsill',  'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg'),
    ('street scene',       'https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/Kitano_Street_Kobe01s5s4110.jpg/320px-Kitano_Street_Kobe01s5s4110.jpg'),
    ('food display',       'https://upload.wikimedia.org/wikipedia/commons/thumb/6/6d/Good_Food_Display_-_NCI_Visuals_Online.jpg/320px-Good_Food_Display_-_NCI_Visuals_Online.jpg'),
    ('ant close-up',       'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg'),
    ('dog breeds',         'https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Dog_Breeds.jpg/320px-Dog_Breeds.jpg'),
]

print('Downloading and encoding image collection...')
collection_images    = []
collection_labels    = []
collection_embeddings = []

for label, url in image_collection:
    filename = f'collection_{label.replace(" ", "_")}.jpg'
    urllib.request.urlretrieve(url, filename)
    img = Image.open(filename).convert('RGB')
    collection_images.append(img)
    collection_labels.append(label)
    collection_embeddings.append(get_image_embedding(img))

# Stack all embeddings into one matrix
collection_embeddings = torch.cat(collection_embeddings, dim=0)  # (N, 512)
print(f'Collection ready: {len(collection_images)} images encoded.')

In [ ]:
def text_image_search(query, top_k=3):
    """
    Find the top-k images in the collection that best match a text query.
    """
    # Encode the query
    query_emb = get_text_embedding(query)   # (1, 512)
    
    # Compute similarity to every image in the collection
    similarities = (query_emb @ collection_embeddings.T).squeeze(0)  # (N,)
    similarities = similarities.cpu().numpy()
    
    # Rank by similarity
    ranked = np.argsort(similarities)[::-1][:top_k]
    
    # Display results
    fig, axes = plt.subplots(1, top_k, figsize=(5 * top_k, 5))
    if top_k == 1:
        axes = [axes]
    
    for rank, (ax, idx) in enumerate(zip(axes, ranked)):
        ax.imshow(collection_images[idx])
        ax.set_title(
            f'Rank {rank+1}: {collection_labels[idx]}\n'
            f'Similarity: {similarities[idx]:.3f}',
            fontsize=10
        )
        ax.axis('off')
    
    plt.suptitle(f'Query: "{query}"', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Try some queries
text_image_search('a furry animal')
text_image_search('urban environment with people')
text_image_search('something you would find in a kitchen')

In [ ]:
# Try queries that require abstract understanding
text_image_search('nature at a small scale')
text_image_search('a place where humans gather')
text_image_search('something alive and moving')

## 4. Image-to-text retrieval

The relationship works in both directions. Given an image, we can find 
which text descriptions best match it — essentially asking the model 
to caption or describe an image by choosing from candidates.

In [ ]:
def image_text_search(image, candidate_descriptions, title=None):
    """
    Find which text descriptions best match a given image.
    """
    img_emb   = get_image_embedding(image)
    txt_embs  = get_text_embedding(candidate_descriptions)
    
    sims  = (img_emb @ txt_embs.T).squeeze(0).cpu().numpy()
    probs = F.softmax(torch.tensor(sims) * 100, dim=0).numpy()
    
    ranked = np.argsort(probs)[::-1]
    
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(14, 5))
    ax_img.imshow(image)
    ax_img.axis('off')
    if title:
        ax_img.set_title(title)
    
    sorted_labels = [candidate_descriptions[i] for i in ranked]
    sorted_probs  = [probs[i] * 100 for i in ranked]
    colors = ['#2196F3' if i == 0 else '#90CAF9' for i in range(len(sorted_labels))]
    
    ax_bar.barh(sorted_labels[::-1], sorted_probs[::-1], color=colors[::-1])
    ax_bar.set_xlabel('Match score (%)')
    ax_bar.set_title('Which description fits best?')
    for i, prob in enumerate(sorted_probs[::-1]):
        ax_bar.text(prob + 0.3, i, f'{prob:.1f}%', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()


# Can CLIP pick the right description from a set of candidates?
candidates = [
    'a dog running on a beach',
    'a golden retriever sitting in grass',
    'a cat sleeping indoors',
    'a wolf in a snowy forest',
    'a puppy playing with a toy',
    'a dog portrait in natural light',
]
image_text_search(dog_image, candidates, title='Which description fits this image?')

## 5. Try your own queries

Add your own images to the collection and try your own search queries. 
This is most interesting when the images come from a domain you care about.

In [ ]:
# Upload your own images to search over
from google.colab import files

print('Upload one or more images to add to the collection:')
uploaded = files.upload()

for filename in uploaded.keys():
    img = Image.open(filename).convert('RGB')
    collection_images.append(img)
    collection_labels.append(filename)
    collection_embeddings_new = get_image_embedding(img)
    collection_embeddings = torch.cat([collection_embeddings, collection_embeddings_new], dim=0)
    print(f'Added {filename} to collection. Total: {len(collection_images)} images.')

In [ ]:
# Now search with your own queries
my_query = 'your search query here'
text_image_search(my_query, top_k=3)

## 6. The limits of zero-shot understanding

CLIP is impressive, but it has important limitations. Let's probe a few of them.

In [ ]:
# Spatial reasoning — CLIP often struggles with left/right, above/below
spatial_queries = [
    'the dog is on the left side of the image',
    'the dog is on the right side of the image',
    'the dog is in the centre of the image',
]
print('Spatial reasoning test (CLIP often gets this wrong):')
image_text_search(dog_image, spatial_queries)

# Counting — CLIP is not reliable at counting objects
counting_queries = [
    'one dog in the image',
    'two dogs in the image',
    'three or more dogs in the image',
]
print('\nCounting test:')
image_text_search(dog_image, counting_queries)
print('\nNote: CLIP was not designed for precise spatial or numerical reasoning.')
print('These capabilities require different architectures or fine-tuning.')

## 7. What can you do with this?

CLIP opens up a range of applications that are difficult or impossible 
with traditional supervised models:

**Zero-shot classification** — classify images into categories you define 
in natural language, with no labelled training data. Useful when you have 
a clear description of your categories but limited or no labelled examples.

**Image search** — build a semantic search engine over an image collection. 
Users search with natural language queries rather than keywords or tags. 
Particularly powerful for large unlabelled datasets.

**Data annotation assistance** — use CLIP to do a first-pass classification 
of unlabelled data before human review. Dramatically reduces annotation time.

**Multimodal retrieval** — find images that match text, or find text that 
matches images. Useful in e-commerce (find products from a description), 
medical literature (find relevant images for a clinical description), 
or scientific research (find figures relevant to a hypothesis).

**Conditioning generative models** — CLIP embeddings are used to condition 
diffusion models, enabling text-to-image generation. The Stable Diffusion 
model in Demo 4 uses CLIP's text encoder internally.

---

### Think about your project

1. Do you have a domain with images but limited or no labels? 
   Could zero-shot classification give you a useful starting point?
2. Is there a problem in your domain where searching images with 
   natural language queries would be valuable?
3. How would you evaluate whether CLIP's zero-shot performance 
   is good enough for your task — or whether you need fine-tuning 
   on domain-specific data?

---

## End of Week 1 demos

You have now seen five major capability areas of modern deep learning for images:
classification, detection, segmentation, generation, and vision-language understanding.

Before next week, write down at least two project ideas that were sparked 
by something you saw today. They do not need to be fully formed — just a 
domain and a problem. You will develop them further over the next few weeks.